In [9]:
import pandas as pd
import numpy as np


In [10]:


cols = ["date", "text"]

df = pd.read_csv("Bitcoin_tweets.csv", usecols=cols)

/var/folders/99/7sjp9rrj0hnbskrtn40bmxlm0000gn/T/ipykernel_70703/2149833329.py:3: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Bitcoin_tweets.csv", usecols=cols)


In [11]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])  # drop rows where date conversion failed

In [12]:
df = df[['date', 'text']]

In [13]:
df.head()

,date,text
0,2021-02-10 23:59:04,Blue Ridge Bank shares halted by NYSE after #b...
1,2021-02-10 23:58:48,"😎 Today, that's this #Thursday, we will do a ""..."
2,2021-02-10 23:54:48,"Guys evening, I have read this article about B..."
3,2021-02-10 23:54:33,$BTC A big chance in a billion! Price: \487264...
4,2021-02-10 23:54:06,This network is secured by 9 508 nodes as of t...


In [15]:
df['date'] = df['date'].dt.date


In [16]:
import yfinance as yf

btc = yf.download("BTC-USD", start="2021-02-01", end="2023-01-10")
btc.reset_index(inplace=True)

btc.head()

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open,Volume
Ticker,,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
0,2021-02-01,33537.175781,34638.214844,32384.228516,33114.578125,61400400660
1,2021-02-02,35510.289062,35896.882812,33489.218750,33533.199219,63088585433
2,2021-02-03,37472.089844,37480.187500,35443.984375,35510.820312,61166818159
3,2021-02-04,36926.066406,38592.175781,36317.500000,37475.105469,68838074392
4,2021-02-05,38144.308594,38225.906250,36658.761719,36931.546875,58598066402


In [17]:
btc.columns = btc.columns.get_level_values(0)


In [18]:
btc.head()


Price,Date,Close,High,Low,Open,Volume
0,2021-02-01,33537.175781,34638.214844,32384.228516,33114.578125,61400400660
1,2021-02-02,35510.289062,35896.882812,33489.218750,33533.199219,63088585433
2,2021-02-03,37472.089844,37480.187500,35443.984375,35510.820312,61166818159
3,2021-02-04,36926.066406,38592.175781,36317.500000,37475.105469,68838074392
4,2021-02-05,38144.308594,38225.906250,36658.761719,36931.546875,58598066402


Install & Import Sentiment Model

In [19]:
import nltk
nltk.download('vader_lexicon')

from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/m2/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [20]:
def get_sentiment(text):
    return sia.polarity_scores(text)['compound']

In [ ]:
df['sentiment'] = df['text'].map(get_sentiment)

Aggregate by date:


In [ ]:
daily_sentiment = df.groupby(df['date'])['sentiment'].mean().compute()
daily_sentiment = daily_sentiment.reset_index()
daily_sentiment.columns = ['date', 'avg_sentiment']

In [ ]:

daily_sentiment.to_csv("daily_sentiment.csv", index=False)

In [ ]:
daily_sentiment = pd.read_csv("daily_sentiment.csv")

In [ ]:
daily_sentiment.head()


In [ ]:
btc.dtypes

In [ ]:
daily_sentiment['date'] = pd.to_datetime(daily_sentiment['date'])

In [ ]:
daily_sentiment.dtypes

In [ ]:
merged = pd.merge(
    daily_sentiment,
    btc,
    left_on="date",
    right_on="Date"
)

In [ ]:
merged.head()

In [ ]:
merged = merged.sort_values("date")
merged = merged.reset_index(drop=True)

today’s sentiment predict tomorrow’s price movement?
(Closet+1​−Closet​​)/Closet
(“Store tomorrow’s percentage price change in today’s row.”)

In [ ]:
merged['next_day_return'] = merged['Close'].pct_change().shift(-1)

In [ ]:
merged.head()

In [ ]:
merged=merged.dropna()

In [ ]:
merged['target'] = (merged['next_day_return'] > 0).astype(int)

In [ ]:
merged['target'].value_counts()

In [ ]:
merged.info()


In [ ]:
merged.drop(columns=["date"], inplace=True)

In [ ]:
merged.describe()

In [ ]:
import matplotlib.pyplot as plt

plt.hist(merged['avg_sentiment'], bins=30)
plt.title("Distribution of Average Daily Sentiment")
plt.xlabel("Sentiment Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.hist(merged['next_day_return'], bins=30)
plt.title("Distribution of Next Day Returns")
plt.xlabel("Return")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.scatter(merged['avg_sentiment'], merged['next_day_return'])
plt.xlabel("Average Sentiment")
plt.ylabel("Next Day Return")
plt.title("Sentiment vs Next Day Return")
plt.show()

In [ ]:
merged.sort_values("next_day_return", ascending=False).head(1)

Correlation WITH outlier

In [ ]:
merged[['avg_sentiment', 'next_day_return']].corr()

Correlation WITHOUT extreme moves (>20%)

In [ ]:
filtered = merged[merged['next_day_return'].abs() < 0.2]
filtered[['avg_sentiment', 'next_day_return']].corr()

In [ ]:
import seaborn as sns

plt.figure(figsize=(8,6))
sns.heatmap(
    merged[['avg_sentiment', 'next_day_return', 'Volume']].corr(),
    annot=True,
    cmap="coolwarm"
)
plt.title("Correlation Matrix")
plt.show()

Sentiment Over Time


In [ ]:
plt.plot(merged['Date'], merged['avg_sentiment'])
plt.title("Sentiment Over Time")
plt.xticks(rotation=45)
plt.show()

BTC Price Over Time

In [ ]:
plt.plot(merged['Date'], merged['Close'])
plt.title("BTC Price Over Time")
plt.xticks(rotation=45)
plt.show()

creating  new lag sentiment features to capture delayed reaction


-Day Rolling Sentiment
We compute a 3-day and 7-day moving average of daily sentiment:
This represents the average sentiment over the last three days.
used to capture trend

Measures percentage price change from previous day.  

Measures intraday price fluctuation relative to opening price.  

Measures change in trading activity from previous day.  



In [ ]:
merged['return_1d'] = merged['Close'].pct_change()
merged['volatility'] = (merged['High'] - merged['Low']) / merged['Open']
merged['volume_change'] = merged['Volume'].pct_change()

Log Transform to handle volume data

In [ ]:
merged['log_volume'] = np.log1p(merged['Volume'])

In [ ]:
merged.isnull().sum()

In [ ]:
merged = merged.dropna()

Define Feature Matrix. 

In [ ]:
features = [
    'avg_sentiment',
    'return_1d',
    'volatility',
    'volume_change',
    'log_volume'
]

X = merged[features]
y = merged['target']

Time-Based Train–Test Split

To avoid look-ahead bias, the dataset is split chronologically.

All data before 1 august 2022 is used for training,
and data from 1 august 2022 onward is used for testing.

In [ ]:
train = merged[merged['Date'] < '2022-08-01']
test  = merged[merged['Date'] >= '2022-08-01']
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

X_train = train[features]
y_train = train['target']

X_test = test[features]
y_test = test['target']

In [ ]:
X_train.head()

MODEL training

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

rf.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

In [ ]:
importance = pd.Series(rf.feature_importances_, index=features)
importance.sort_values(ascending=False)

In [ ]:
import joblib

joblib.dump(rf, "btc_sentiment_model.pkl")

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model= LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

In [ ]:
y_prep=model.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_prep))
print("\nClassification Report:\n", classification_report(y_test, y_prep))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_prep))

In [ ]:
from xgboost import XGBClassifier

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

In [ ]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))

In [ ]:
importance = pd.Series(xgb.feature_importances_, index=features)
print(importance.sort_values(ascending=False))